# 03 - Loss Landscape Comparisons

Side-by-side contour plots of P6 landscape slices for runs that ran the deep-probe set.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from playground.runs_io import load_sweep

In [ ]:
EXPERIMENT = "smoke_sweep"
runs_dir = Path.cwd().parent / "runs" / EXPERIMENT
if not runs_dir.exists():
    print(f"No runs at {runs_dir}.")
    df = pd.DataFrame()
else:
    df = load_sweep(runs_dir)
    print(f"loaded {len(df)} runs")

In [ ]:
landscape_runs = []
if not df.empty:
    for _, row in df.iterrows():
        p = Path(row["run_dir"]) / "probes" / "landscape.npz"
        if p.exists():
            landscape_runs.append((row, p))
print(f"runs with landscape probes: {len(landscape_runs)}")

In [ ]:
if landscape_runs:
    fig, axes = plt.subplots(1, len(landscape_runs), figsize=(5 * len(landscape_runs), 4), squeeze=False)
    probe_prefixes = ("margin_", "grad_", "adversarial_", "calibration_", "thickness_", "hessian_", "landscape_")
    summary_suffixes = ("_final", "_best")
    skip_cols = {"run_dir", "experiment_name", "run_id", "extra", "description"}
    config_cols = [c for c in df.columns if not c.endswith(summary_suffixes) and not c.startswith(probe_prefixes) and c not in skip_cols]
    varying = [c for c in config_cols if df[c].nunique() > 1 and c != "seed"]
    title_dial = varying[0] if varying else None

    for ax, (row, npz_path) in zip(axes[0], landscape_runs):
        data = np.load(npz_path)
        grid = data["loss_grid"]
        cf = ax.contourf(data["betas"], data["alphas"], grid, levels=30, cmap="viridis")
        fig.colorbar(cf, ax=ax)
        label = f"{title_dial}={row[title_dial]}" if title_dial else "run"
        ax.set_title(f"{label}, seed={row['seed']}")
    plt.show()